In [1]:

import pandas as pd
import re



invoices = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned\invoices.csv")
payments = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned\gateway_payments.csv")



def normalize_invoice_id(invoice_id):
    if pd.isna(invoice_id):
        return None

    
    return re.sub(r"[^A-Z0-9]", "", str(invoice_id).upper())


invoices["normalized_invoice_id"] = (
    invoices["invoice_id"].apply(normalize_invoice_id)
)

payments["normalized_invoice_id"] = (
    payments["invoice_ref"].apply(normalize_invoice_id)
)



payments["status"] = payments["status"].astype(str).str.strip().str.lower()

captured_payments = payments[
    payments["status"] == "captured"
].copy()



paid_invoice_ids = set(
    captured_payments["normalized_invoice_id"].dropna()
)



unpaid_invoices = invoices[
    ~invoices["normalized_invoice_id"].isin(paid_invoice_ids)
].copy()

columns_to_display = [
    "invoice_id",
    "invoice_date",
    "booking_id",
    "patient_id",
    "city",
    "channel",
    "amount_rs"
]

print("\nUNPAID INVOICES")
print(unpaid_invoices[columns_to_display].to_string(index=False))

# --------------------------------------------------
# 7. Calculate the count and total amount
# --------------------------------------------------

unpaid_count = unpaid_invoices["invoice_id"].nunique()

unpaid_amount = pd.to_numeric(
    unpaid_invoices["amount_rs"], errors="coerce"
).sum()

print("\nUNPAID INVOICE SUMMARY")
print("----------------------")
print("Number of unpaid invoices:", unpaid_count)
print("Total unpaid amount (Rs):", round(unpaid_amount, 2))



unpaid_invoices[columns_to_display].to_csv(
    "unpaid_invoices.csv", index=False
)

print("\nSaved results to unpaid_invoices.csv")



UNPAID INVOICES
     invoice_id        invoice_date            booking_id  patient_id      city channel  amount_rs
INV-2025-000032 2025-01-01 00:00:00  BK-AHM-202501-004831 PAHM0033938 Ahmedabad    home      720.0
INV-2025-000046 2025-01-01 00:00:00  BK-BLR-202501-004958 PBLR0043174 Bengaluru    home     1050.0
INV-2025-000055 2025-01-01 00:00:00  BK-BLR-202501-004801 PBLR0017058 Bengaluru    home     1600.0
INV-2025-000084 2025-01-01 00:00:00  BK-AHM-202501-004236 PAHM0030173 Ahmedabad    home      800.0
INV-2025-000102 2025-01-01 00:00:00  BK-BLR-202501-004127 PBLR0019746 Bengaluru    home      720.0
INV-2025-000125 2025-01-01 00:00:00  BK-BLR-202501-003733 PBLR0003757 Bengaluru    home     1149.0
INV-2025-000169 2025-01-01 00:00:00  BK-HYD-202501-003635 PHYD0027999 Hyderabad  clinic      350.0
INV-2025-000181 2025-01-01 00:00:00  BK-BLR-202501-006820 PBLR0044062 Bengaluru  clinic     1100.0
INV-2025-000189 2025-01-01 00:00:00  BK-HYD-202501-004313 PHYD0006895 Hyderabad  clinic     

In [2]:
def normalize_invoice_id(invoice_id):
    if pd.isna(invoice_id):
        return None

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(invoice_id).upper()
    )


invoices["normalized_invoice_id"] = (
    invoices["invoice_id"].apply(normalize_invoice_id)
)

payments["normalized_invoice_id"] = (
    payments["invoice_ref"].apply(normalize_invoice_id)
)

# --------------------------------------------------
# 3. Create the set of valid invoice IDs
# --------------------------------------------------

valid_invoice_ids = set(
    invoices["normalized_invoice_id"].dropna()
)

# --------------------------------------------------
# 4. Find payments that match no invoice
# --------------------------------------------------

unmatched_payments = payments[
    payments["normalized_invoice_id"].isna()
    | ~payments["normalized_invoice_id"].isin(valid_invoice_ids)
].copy()

# --------------------------------------------------
# 5. Prepare the results
# --------------------------------------------------

display_columns = [
    "payment_id",
    "invoice_ref",
    "amount_rs",
    "status",
    "created_at_epoch",
    "method",
    "settlement_id"
]

print("\nPAYMENTS THAT MATCH NO INVOICE")
print("=" * 60)

if unmatched_payments.empty:
    print("No unmatched payments found.")
else:
    print(
        unmatched_payments[display_columns]
        .to_string(index=False)
    )

# --------------------------------------------------
# 6. Calculate unmatched payment metrics
# --------------------------------------------------

unmatched_count = unmatched_payments["payment_id"].nunique()

unmatched_amount = pd.to_numeric(
    unmatched_payments["amount_rs"],
    errors="coerce"
).sum()

print("\nUNMATCHED PAYMENT SUMMARY")
print("-" * 40)
print("Unmatched payment rows:", len(unmatched_payments))
print("Unique payment IDs:", unmatched_count)
print("Total unmatched amount (Rs):", round(unmatched_amount, 2))

# --------------------------------------------------
# 7. Separate captured unmatched payments
# --------------------------------------------------

payments_status = unmatched_payments["status"].astype(str).str.lower()

captured_unmatched = unmatched_payments[
    payments_status == "captured"
]

print("\nCAPTURED UNMATCHED PAYMENTS")
print("-" * 40)
print("Captured unmatched rows:", len(captured_unmatched))
print(
    "Captured unmatched amount (Rs):",
    round(
        pd.to_numeric(
            captured_unmatched["amount_rs"],
            errors="coerce"
        ).sum(),
        2
    )
)

# --------------------------------------------------
# 8. Export results for investigation
# --------------------------------------------------

unmatched_payments[display_columns].to_csv(
    "unmatched_payments.csv",
    index=False
)

print("\nSaved results to unmatched_payments.csv")



PAYMENTS THAT MATCH NO INVOICE
    payment_id      invoice_ref  amount_rs   status  created_at_epoch     method settlement_id
pay_00000000b6  INV-2025-900181     1100.0 captured        1736518218        upi setl_6781b480
pay_0000000132  INV-2025-900307     3999.0 captured        1736176497        upi setl_677c6e80
pay_000000013e  INV-2025-900320      800.0 captured        1735820463       cash setl_67772880
pay_0000000194  INV-2025-900404      120.0 captured        1735917478        upi setl_67787a00
pay_00000001ac  INV-2025-900427     4249.0 captured        1736944770       card setl_67884c00
pay_00000001ef  INV-2025-900492     1999.0 captured        1735756103        upi setl_6775d700
pay_0000000247  INV-2025-900581      800.0 captured        1736171873       card setl_677c6e80
pay_0000000297  INV-2025-900662      330.0 captured        1735840713 netbanking setl_67772880
pay_00000003cf  INV-2025-900964     2250.0 captured        1735820771       card setl_67772880
pay_0000000485  IN

In [ ]:
# Change these dates if your Q2 is different
quarter_start = pd.Timestamp("2025-04-01")
quarter_end = pd.Timestamp("2025-06-30 23:59:59")

# --------------------------------------------------
# 2. Normalize invoice IDs
# --------------------------------------------------

def normalize_invoice_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["normalized_invoice_id"] = (
    invoices["invoice_id"].apply(normalize_invoice_id)
)

payments["normalized_invoice_id"] = (
    payments["invoice_ref"].apply(normalize_invoice_id)
)

# --------------------------------------------------
# 3. Convert dates
# --------------------------------------------------

invoices["invoice_date"] = pd.to_datetime(
    invoices["invoice_date"], errors="coerce"
)

# Unix timestamps are in seconds; convert to India time
payments["payment_date"] = pd.to_datetime(
    pd.to_numeric(payments["created_at_epoch"], errors="coerce"),
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)

payments["status"] = (
    payments["status"].astype(str).str.strip().str.lower()
)

invoices["amount_rs"] = pd.to_numeric(
    invoices["amount_rs"], errors="coerce"
)

payments["amount_rs"] = pd.to_numeric(
    payments["amount_rs"], errors="coerce"
)

# --------------------------------------------------
# 4. Select invoices issued during Q2
# --------------------------------------------------

q2_invoices = invoices[
    invoices["invoice_date"].between(
        quarter_start, quarter_end
    )
].copy()

# --------------------------------------------------
# 5. Select successfully captured payments
# --------------------------------------------------

captured = payments[
    payments["status"] == "captured"
].copy()

# Remove duplicate payment IDs, if any.
# Keep the first occurrence and investigate duplicates separately.
captured = captured.drop_duplicates(
    subset=["payment_id"], keep="first"
)

# --------------------------------------------------
# 6. Match payments to invoices
# --------------------------------------------------

matched = captured.merge(
    invoices[
        ["invoice_id", "normalized_invoice_id", "invoice_date"]
    ],
    on="normalized_invoice_id",
    how="inner",
    suffixes=("_payment", "_invoice")
)

# Keep payments linked to Q2 invoices
q2_invoice_ids = set(q2_invoices["normalized_invoice_id"])

q2_matched = matched[
    matched["normalized_invoice_id"].isin(q2_invoice_ids)
].copy()

# --------------------------------------------------
# 7. Classify payment timing
# --------------------------------------------------

paid_by_close = q2_matched[
    q2_matched["payment_date"] <= quarter_end
]

paid_after_close = q2_matched[
    q2_matched["payment_date"] > quarter_end
]

# Aggregate payment amounts at invoice level
paid_before = (
    paid_by_close.groupby("normalized_invoice_id")["amount_rs"]
    .sum()
)

paid_later = (
    paid_after_close.groupby("normalized_invoice_id")["amount_rs"]
    .sum()
)

# --------------------------------------------------
# 8. Build invoice-level reconciliation
# --------------------------------------------------

q2_invoices["paid_by_close_rs"] = (
    q2_invoices["normalized_invoice_id"]
    .map(paid_before).fillna(0)
)

q2_invoices["paid_after_close_rs"] = (
    q2_invoices["normalized_invoice_id"]
    .map(paid_later).fillna(0)
)

q2_invoices["outstanding_at_close_rs"] = (
    q2_invoices["amount_rs"]
    - q2_invoices["paid_by_close_rs"]
).clip(lower=0)

q2_invoices["timing_paid_later_rs"] = (
    q2_invoices[["paid_after_close_rs", "outstanding_at_close_rs"]]
    .min(axis=1)
)

# Invoices with payments after quarter-end
timing_invoices = q2_invoices[
    q2_invoices["timing_paid_later_rs"] > 0
].copy()

# --------------------------------------------------
# 9. Calculate timing-gap summary
# --------------------------------------------------

invoiced = q2_invoices["amount_rs"].sum()
collected_by_close = q2_invoices["paid_by_close_rs"].sum()
paid_later = q2_invoices["timing_paid_later_rs"].sum()

unpaid_at_close = (
    q2_invoices["outstanding_at_close_rs"]
    .sum()
)

print("\nQ2 TIMING RECONCILIATION")
print("=" * 50)
print("Quarter:", quarter_start.date(), "to", quarter_end.date())
print("Invoices raised (count):", q2_invoices["invoice_id"].nunique())
print("Amount invoiced (Rs):", round(invoiced, 2))
print("Payments linked by quarter-end (Rs):",
      round(collected_by_close, 2))
print("Of the outstanding amount, paid after quarter-end (Rs):",
      round(paid_later, 2))
print("Outstanding at quarter-end (Rs):",
      round(unpaid_at_close, 2))

# --------------------------------------------------
# 10. List invoices paid after quarter-end
# --------------------------------------------------

print("\nINVOICES WITH LATER PAYMENTS")
print(
    timing_invoices[
        [
            "invoice_id",
            "invoice_date",
            "amount_rs",
            "paid_by_close_rs",
            "paid_after_close_rs",
            "timing_paid_later_rs"
        ]
    ].to_string(index=False)
)# Change these dates if your Q2 is different
quarter_start = pd.Timestamp("2025-04-01")
quarter_end = pd.Timestamp("2025-06-30 23:59:59")

# --------------------------------------------------
# 2. Normalize invoice IDs
# --------------------------------------------------

def normalize_invoice_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["normalized_invoice_id"] = (
    invoices["invoice_id"].apply(normalize_invoice_id)
)

payments["normalized_invoice_id"] = (
    payments["invoice_ref"].apply(normalize_invoice_id)
)

# --------------------------------------------------
# 3. Convert dates
# --------------------------------------------------

invoices["invoice_date"] = pd.to_datetime(
    invoices["invoice_date"], errors="coerce"
)

# Unix timestamps are in seconds; convert to India time
payments["payment_date"] = pd.to_datetime(
    pd.to_numeric(payments["created_at_epoch"], errors="coerce"),
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)

payments["status"] = (
    payments["status"].astype(str).str.strip().str.lower()
)

invoices["amount_rs"] = pd.to_numeric(
    invoices["amount_rs"], errors="coerce"
)

payments["amount_rs"] = pd.to_numeric(
    payments["amount_rs"], errors="coerce"
)

# --------------------------------------------------
# 4. Select invoices issued during Q2
# --------------------------------------------------

q2_invoices = invoices[
    invoices["invoice_date"].between(
        quarter_start, quarter_end
    )
].copy()

# --------------------------------------------------
# 5. Select successfully captured payments
# --------------------------------------------------

captured = payments[
    payments["status"] == "captured"
].copy()

# Remove duplicate payment IDs, if any.
# Keep the first occurrence and investigate duplicates separately.
captured = captured.drop_duplicates(
    subset=["payment_id"], keep="first"
)

# --------------------------------------------------
# 6. Match payments to invoices
# --------------------------------------------------

matched = captured.merge(
    invoices[
        ["invoice_id", "normalized_invoice_id", "invoice_date"]
    ],
    on="normalized_invoice_id",
    how="inner",
    suffixes=("_payment", "_invoice")
)

# Keep payments linked to Q2 invoices
q2_invoice_ids = set(q2_invoices["normalized_invoice_id"])

q2_matched = matched[
    matched["normalized_invoice_id"].isin(q2_invoice_ids)
].copy()

# --------------------------------------------------
# 7. Classify payment timing
# --------------------------------------------------

paid_by_close = q2_matched[
    q2_matched["payment_date"] <= quarter_end
]

paid_after_close = q2_matched[
    q2_matched["payment_date"] > quarter_end
]

# Aggregate payment amounts at invoice level
paid_before = (
    paid_by_close.groupby("normalized_invoice_id")["amount_rs"]
    .sum()
)

paid_later = (
    paid_after_close.groupby("normalized_invoice_id")["amount_rs"]
    .sum()
)

# --------------------------------------------------
# 8. Build invoice-level reconciliation
# --------------------------------------------------

q2_invoices["paid_by_close_rs"] = (
    q2_invoices["normalized_invoice_id"]
    .map(paid_before).fillna(0)
)

q2_invoices["paid_after_close_rs"] = (
    q2_invoices["normalized_invoice_id"]
    .map(paid_later).fillna(0)
)

q2_invoices["outstanding_at_close_rs"] = (
    q2_invoices["amount_rs"]
    - q2_invoices["paid_by_close_rs"]
).clip(lower=0)

q2_invoices["timing_paid_later_rs"] = (
    q2_invoices[["paid_after_close_rs", "outstanding_at_close_rs"]]
    .min(axis=1)
)

# Invoices with payments after quarter-end
timing_invoices = q2_invoices[
    q2_invoices["timing_paid_later_rs"] > 0
].copy()

# --------------------------------------------------
# 9. Calculate timing-gap summary
# --------------------------------------------------

invoiced = q2_invoices["amount_rs"].sum()
collected_by_close = q2_invoices["paid_by_close_rs"].sum()
paid_later = q2_invoices["timing_paid_later_rs"].sum()

unpaid_at_close = (
    q2_invoices["outstanding_at_close_rs"]
    .sum()
)

print("\nQ2 TIMING RECONCILIATION")
print("=" * 50)
print("Quarter:", quarter_start.date(), "to", quarter_end.date())
print("Invoices raised (count):", q2_invoices["invoice_id"].nunique())
print("Amount invoiced (Rs):", round(invoiced, 2))
print("Payments linked by quarter-end (Rs):",
      round(collected_by_close, 2))
print("Of the outstanding amount, paid after quarter-end (Rs):",
      round(paid_later, 2))
print("Outstanding at quarter-end (Rs):",
      round(unpaid_at_close, 2))

# --------------------------------------------------
# 10. List invoices paid after quarter-end
# --------------------------------------------------

print("\nINVOICES WITH LATER PAYMENTS")
print(
    timing_invoices[
        [
            "invoice_id",
            "invoice_date",
            "amount_rs",
            "paid_by_close_rs",
            "paid_after_close_rs",
            "timing_paid_later_rs"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# 11. Export reconciliation
# --------------------------------------------------

q2_invoices.to_csv("q2_invoice_timing_reconciliation.csv",
                   index=False)

timing_invoices.to_csv("q2_late_paid_invoices.csv",
                       index=False)

print("\nSaved:")
print("- q2_invoice_timing_reconciliation.csv")
print("- q2_late_paid_invoices.csv")


# --------------------------------------------------
# 11. Export reconciliation
# --------------------------------------------------

q2_invoices.to_csv("q2_invoice_timing_reconciliation.csv",
                   index=False)

timing_invoices.to_csv("q2_late_paid_invoices.csv",
                       index=False)

print("\nSaved:")
print("- q2_invoice_timing_reconciliation.csv")
print("- q2_late_paid_invoices.csv")



Q2 TIMING RECONCILIATION
Quarter: 2025-04-01 to 2025-06-30
Invoices raised (count): 77915
Amount invoiced (Rs): 118105523.0
Payments linked by quarter-end (Rs): 109466057.0
Of the outstanding amount, paid after quarter-end (Rs): 4407518.0
Outstanding at quarter-end (Rs): 9088446.0

INVOICES WITH LATER PAYMENTS
     invoice_id invoice_date  amount_rs  paid_by_close_rs  paid_after_close_rs  timing_paid_later_rs
INV-2025-144477   2025-06-16     5148.0               0.0               5148.0                5148.0
INV-2025-144568   2025-06-16      550.0               0.0                550.0                 550.0
INV-2025-144679   2025-06-16      420.0               0.0                420.0                 420.0
INV-2025-144688   2025-06-16      270.0               0.0                270.0                 270.0
INV-2025-144763   2025-06-16     1550.0               0.0               1550.0                1550.0
INV-2025-144824   2025-06-16     1050.0               0.0               1050.0   

In [ ]:
duplicate_rows = payments[
    payments.duplicated(subset=["payment_id"], keep=False)
].sort_values("payment_id")

print("DUPLICATE PAYMENT RECORDS")
print("=" * 40)

if duplicate_rows.empty:
    print("No duplicate payment IDs found.")
else:
    print(duplicate_rows.to_string(index=False))

# Calculate duplicate summary
duplicate_ids = duplicate_rows["payment_id"].nunique()

print("\nDuplicate payment IDs:", duplicate_ids)
print("Rows involved in duplicates:", len(duplicate_rows))

# Save records for investigation
duplicate_rows.to_csv("duplicate_payments.csv", index=False)
print("Saved duplicate_payments.csv")


In [ ]:
settlements = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned\settlements.csv")

# Convert numeric columns
for col in ["amount_rs"]:
    payments[col] = pd.to_numeric(payments[col], errors="coerce")

for col in ["amount_rs", "fees_rs", "tax_rs"]:
    settlements[col] = pd.to_numeric(
        settlements[col], errors="coerce"
    )

# Keep captured payments
payments["status"] = (
    payments["status"].astype(str).str.strip().str.lower()
)

captured = payments[payments["status"] == "captured"].copy()

# Summarize captured payments by settlement ID
payment_summary = (
    captured.groupby("settlement_id", dropna=False)
    .agg(
        payment_count=("payment_id", "nunique"),
        payment_amount_rs=("amount_rs", "sum")
    )
    .reset_index()
)

# Compare with settlement records
reconciliation = settlements.merge(
    payment_summary,
    on="settlement_id",
    how="left"
)

reconciliation["payment_count"] = (
    reconciliation["payment_count"].fillna(0).astype(int)
)

reconciliation["payment_amount_rs"] = (
    reconciliation["payment_amount_rs"].fillna(0)
)

# Calculate expected net settlement
reconciliation["expected_net_rs"] = (
    reconciliation["payment_amount_rs"]
    - reconciliation["fees_rs"]
    - reconciliation["tax_rs"]
)

# Compare payment total with settlement amount
reconciliation["gross_difference_rs"] = (
    reconciliation["payment_amount_rs"]
    - reconciliation["amount_rs"]
)

# Compare expected net with recorded settlement amount
reconciliation["net_difference_rs"] = (
    reconciliation["expected_net_rs"]
    - reconciliation["amount_rs"]
)

print("\nSETTLEMENT RECONCILIATION")
print("=" * 80)

print(
    reconciliation[
        [
            "settlement_id",
            "payment_count",
            "payment_amount_rs",
            "amount_rs",
            "fees_rs",
            "tax_rs",
            "expected_net_rs",
            "gross_difference_rs",
            "net_difference_rs"
        ]
    ].to_string(index=False)
)

# Save results
reconciliation.to_csv(
    "settlement_reconciliation.csv",
    index=False
)

print("\nSaved settlement_reconciliation.csv")


In [ ]:

import pandas as pd
import re
from pathlib import Path

# ============================================================
# KALPA HEALTH: INVOICE VS COLLECTIONS RECONCILIATION
# ============================================================

OUTPUT_FOLDER = Path("reconciliation_output")
OUTPUT_FOLDER.mkdir(exist_ok=True)

INVOICE_FILE = r"C:\Users\Diksha\Documents\cleaned\invoices.csv"
PAYMENT_FILE = r"C:\Users\Diksha\Documents\cleaned\gateway_payments.csv"
SETTLEMENT_FILE = r"C:\Users\Diksha\Documents\cleaned\settlements.csv"

# Quarter to analyse for the quarter-end outstanding report
QUARTER_START = pd.Timestamp("2025-04-01")
QUARTER_END = pd.Timestamp("2025-06-30 23:59:59")

# Differences up to one paisa are treated as rounding differences
TOLERANCE_RS = 0.01

# Payments are counted as collected only when status is captured
CAPTURED_STATUS = {"captured"}

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def normalize_id(value):
    """Remove punctuation and spaces so equivalent IDs match."""
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def to_number(series):
    """Convert a column to numbers; invalid or blank values become zero."""
    return pd.to_numeric(series, errors="coerce").fillna(0)


def epoch_to_local(series):
    """Convert Unix timestamps in seconds to India local time."""
    numeric = pd.to_numeric(series, errors="coerce")
    utc_dates = pd.to_datetime(
        numeric, unit="s", errors="coerce", utc=True
    )
    return utc_dates.dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)


def save_csv(df, filename):
    """Save a report to the output folder."""
    path = OUTPUT_FOLDER / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


def money(value):
    """Display a number as Indian rupees."""
    return f"₹{value:,.2f}"


def require_columns(df, required, filename):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(
            f"{filename} is missing required columns: {missing}"
        )


# ============================================================
# 3. LOAD THE THREE DATASETS
# ============================================================

invoices = pd.read_csv(INVOICE_FILE)
payments = pd.read_csv(PAYMENT_FILE)
settlements = pd.read_csv(SETTLEMENT_FILE)

require_columns(
    invoices,
    ["invoice_id", "invoice_date", "amount_rs"],
    "invoices.csv"
)

require_columns(
    payments,
    [
        "payment_id",
        "invoice_ref",
        "amount_rs",
        "status",
        "created_at_epoch",
        "settlement_id"
    ],
    "gateway_payments.csv"
)

require_columns(
    settlements,
    [
        "settlement_id",
        "amount_rs",
        "fees_rs",
        "tax_rs",
        "status",
        "created_at_epoch"
    ],
    "settlements.csv"
)

# Clean monetary columns
invoices["amount_rs"] = to_number(invoices["amount_rs"])
payments["amount_rs"] = to_number(payments["amount_rs"])
settlements["amount_rs"] = to_number(settlements["amount_rs"])
settlements["fees_rs"] = to_number(settlements["fees_rs"])
settlements["tax_rs"] = to_number(settlements["tax_rs"])

# Normalize invoice IDs before matching
invoices["invoice_key"] = invoices["invoice_id"].apply(normalize_id)
payments["invoice_key"] = payments["invoice_ref"].apply(normalize_id)

# Clean statuses
payments["status_clean"] = (
    payments["status"].astype("string").str.strip().str.lower()
)
settlements["status_clean"] = (
    settlements["status"].astype("string").str.strip().str.lower()
)

# Convert dates
invoices["invoice_date"] = pd.to_datetime(
    invoices["invoice_date"], errors="coerce"
)
payments["payment_date"] = epoch_to_local(payments["created_at_epoch"])
settlements["settlement_date"] = epoch_to_local(
    settlements["created_at_epoch"]
)

# Only valid invoice IDs can be matched
valid_invoice_keys = set(
    invoices.loc[invoices["invoice_key"] != "", "invoice_key"]
)

captured = payments[
    payments["status_clean"].isin(CAPTURED_STATUS)
].copy()

print("\n" + "=" * 65)
print("DATASET OVERVIEW")
print("=" * 65)
print(f"Invoice rows: {len(invoices):,}")
print(f"Payment rows: {len(payments):,}")
print(f"Captured payment rows: {len(captured):,}")
print(f"Settlement rows: {len(settlements):,}")

# ============================================================
# 4. PAYMENT STATUS SUMMARY
# ============================================================

status_summary = (
    payments.groupby("status_clean", dropna=False)
    .agg(
        payment_rows=("payment_id", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        amount_rs=("amount_rs", "sum")
    )
    .reset_index()
    .rename(columns={"status_clean": "payment_status"})
    .sort_values("amount_rs", ascending=False)
)

save_csv(status_summary, "01_payment_status_summary.csv")

print("\nPAYMENT STATUS SUMMARY")
print(status_summary.to_string(index=False))

# ============================================================
# 5. DUPLICATE PAYMENT ID CHECK
# ============================================================

duplicate_payment_rows = payments[
    payments["payment_id"].notna()
    & payments["payment_id"].duplicated(keep=False)
].sort_values("payment_id")

duplicate_summary = pd.DataFrame([{
    "duplicate_payment_ids": duplicate_payment_rows[
        "payment_id"
    ].nunique(),
    "rows_involved": len(duplicate_payment_rows),
    "amount_involved_rs": duplicate_payment_rows[
        "amount_rs"
    ].sum()
}])

save_csv(duplicate_payment_rows, "02_duplicate_payment_rows.csv")
save_csv(duplicate_summary, "03_duplicate_payment_summary.csv")

print("\nDUPLICATE PAYMENT CHECK")
print(duplicate_summary.to_string(index=False))

# ============================================================
# 6. UNMATCHED PAYMENTS
# ============================================================

# A payment is unmatched if its invoice reference does not
# correspond to any invoice ID after ID normalization.
unmatched_payments = payments[
    (payments["invoice_key"] == "")
    | (~payments["invoice_key"].isin(valid_invoice_keys))
].copy()

unmatched_captured = unmatched_payments[
    unmatched_payments["status_clean"].isin(CAPTURED_STATUS)
].copy()

unmatched_summary = pd.DataFrame([{
    "unmatched_payment_rows": len(unmatched_payments),
    "unmatched_unique_payment_ids": unmatched_payments[
        "payment_id"
    ].nunique(),
    "unmatched_amount_all_statuses_rs": unmatched_payments[
        "amount_rs"
    ].sum(),
    "unmatched_captured_rows": len(unmatched_captured),
    "unmatched_captured_amount_rs": unmatched_captured[
        "amount_rs"
    ].sum()
}])

save_csv(unmatched_payments, "04_unmatched_payments_all.csv")
save_csv(unmatched_captured, "05_unmatched_captured_payments.csv")
save_csv(unmatched_summary, "06_unmatched_payment_summary.csv")

print("\nUNMATCHED PAYMENT SUMMARY")
print(unmatched_summary.to_string(index=False))

# ============================================================
# 7. RECONCILE PAYMENTS AGAINST INVOICES
# ============================================================

# Only captured payments are used to calculate collections.
# Each invoice may have multiple payment rows.
matched_captured = captured[
    (captured["invoice_key"] != "")
    & (captured["invoice_key"].isin(valid_invoice_keys))
].copy()

payments_per_invoice = (
    matched_captured.groupby("invoice_key")
    .agg(
        captured_payment_rows=("payment_id", "size"),
        captured_amount_rs=("amount_rs", "sum"),
        first_payment_date=("payment_date", "min"),
        last_payment_date=("payment_date", "max")
    )
    .reset_index()
)

invoice_reconciliation = invoices.merge(
    payments_per_invoice,
    on="invoice_key",
    how="left"
)

invoice_reconciliation["captured_payment_rows"] = (
    invoice_reconciliation["captured_payment_rows"]
    .fillna(0)
    .astype(int)
)
invoice_reconciliation["captured_amount_rs"] = (
    invoice_reconciliation["captured_amount_rs"].fillna(0)
)

invoice_reconciliation["balance_rs"] = (
    invoice_reconciliation["amount_rs"]
    - invoice_reconciliation["captured_amount_rs"]
)

invoice_reconciliation["invoice_status"] = "Partially paid"

invoice_reconciliation.loc[
    invoice_reconciliation["captured_amount_rs"] <= TOLERANCE_RS,
    "invoice_status"
] = "Unpaid / no captured payment"

invoice_reconciliation.loc[
    invoice_reconciliation["balance_rs"].abs() <= TOLERANCE_RS,
    "invoice_status"
] = "Paid in full"

invoice_reconciliation.loc[
    invoice_reconciliation["balance_rs"] < -TOLERANCE_RS,
    "invoice_status"
] = "Overpaid"

# This column shows the outstanding amount, not a negative balance.
invoice_reconciliation["outstanding_rs"] = (
    invoice_reconciliation["balance_rs"].clip(lower=0)
)

# This column identifies amounts paid beyond the invoice value.
invoice_reconciliation["overpayment_rs"] = (
    (-invoice_reconciliation["balance_rs"]).clip(lower=0)
)

save_csv(invoice_reconciliation, "07_invoice_reconciliation.csv")

invoice_status_summary = (
    invoice_reconciliation.groupby("invoice_status")
    .agg(
        invoice_count=("invoice_id", "size"),
        invoiced_amount_rs=("amount_rs", "sum"),
        captured_amount_rs=("captured_amount_rs", "sum"),
        outstanding_rs=("outstanding_rs", "sum"),
        overpayment_rs=("overpayment_rs", "sum")
    )
    .reset_index()
)

save_csv(invoice_status_summary, "08_invoice_status_summary.csv")

print("\nINVOICE RECONCILIATION SUMMARY")
print(invoice_status_summary.to_string(index=False))

print("\nINVOICE TOTALS")
print("Total invoiced:", money(invoices["amount_rs"].sum()))
print("Captured payments linked to invoices:",
      money(invoice_reconciliation["captured_amount_rs"].sum()))
print("Outstanding invoice balances:",
      money(invoice_reconciliation["outstanding_rs"].sum()))
print("Amounts above invoice values:",
      money(invoice_reconciliation["overpayment_rs"].sum()))

# ============================================================
# 8. SETTLEMENT RECONCILIATION
# ============================================================

# A settlement normally represents a batch of captured payments.
# Compare gross captured payments with the settlement's net amount.
captured["settlement_key"] = captured["settlement_id"].apply(normalize_id)
settlements["settlement_key"] = settlements["settlement_id"].apply(
    normalize_id
)

payments_by_settlement = (
    captured[captured["settlement_key"] != ""]
    .groupby("settlement_key")
    .agg(
        payment_count=("payment_id", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        payment_gross_rs=("amount_rs", "sum"),
        first_payment_date=("payment_date", "min"),
        last_payment_date=("payment_date", "max")
    )
    .reset_index()
)

settlement_reconciliation = settlements.merge(
    payments_by_settlement,
    on="settlement_key",
    how="outer",
    indicator=True,
    suffixes=("_settlement", "_payment")
)

# Missing values indicate that one side has no corresponding record.
for col in [
    "payment_count", "unique_payment_ids", "payment_gross_rs",
    "amount_rs", "fees_rs", "tax_rs"
]:
    if col in settlement_reconciliation.columns:
        settlement_reconciliation[col] = (
            pd.to_numeric(
                settlement_reconciliation[col], errors="coerce"
            ).fillna(0)
        )

settlement_reconciliation["expected_net_rs"] = (
    settlement_reconciliation["payment_gross_rs"]
    - settlement_reconciliation["fees_rs"]
    - settlement_reconciliation["tax_rs"]
)

settlement_reconciliation["gross_difference_rs"] = (
    settlement_reconciliation["payment_gross_rs"]
    - settlement_reconciliation["amount_rs"]
)

settlement_reconciliation["net_difference_rs"] = (
    settlement_reconciliation["expected_net_rs"]
    - settlement_reconciliation["amount_rs"]
)

settlement_reconciliation["reconciliation_status"] = "Reconciled"

settlement_reconciliation.loc[
    settlement_reconciliation["_merge"] == "left_only",
    "reconciliation_status"
] = "Settlement has no matching captured-payment batch"

settlement_reconciliation.loc[
    settlement_reconciliation["_merge"] == "right_only",
    "reconciliation_status"
] = "Captured-payment batch has no settlement record"

settlement_reconciliation.loc[
    (settlement_reconciliation["_merge"] == "both")
    & (
        settlement_reconciliation["net_difference_rs"].abs()
        > TOLERANCE_RS
    ),
    "reconciliation_status"
] = "Net difference needs investigation"

save_csv(
    settlement_reconciliation,
    "09_settlement_reconciliation.csv"
)

settlement_summary = pd.DataFrame([{
    "captured_gross_in_batches_rs":
        settlement_reconciliation["payment_gross_rs"].sum(),
    "settlement_fees_rs":
        settlement_reconciliation["fees_rs"].sum(),
    "settlement_taxes_rs":
        settlement_reconciliation["tax_rs"].sum(),
    "expected_net_after_fees_taxes_rs":
        settlement_reconciliation["expected_net_rs"].sum(),
    "recorded_settlement_amount_rs":
        settlement_reconciliation["amount_rs"].sum(),
    "net_difference_rs":
        settlement_reconciliation["net_difference_rs"].sum(),
    "settlements_needing_investigation":
        (
            settlement_reconciliation["reconciliation_status"]
            == "Net difference needs investigation"
        ).sum(),
    "settlement_records_without_payment_batch":
        (
            settlement_reconciliation["_merge"] == "left_only"
        ).sum(),
    "payment_batches_without_settlement_record":
        (
            settlement_reconciliation["_merge"] == "right_only"
        ).sum()
}])

save_csv(settlement_summary, "10_settlement_summary.csv")

print("\nSETTLEMENT SUMMARY")
print(settlement_summary.to_string(index=False))

print("\nSETTLEMENT STATUS COUNTS")
print(
    settlement_reconciliation["reconciliation_status"]
    .value_counts(dropna=False)
    .to_string()
)

# ============================================================
# 9. MONTHLY INVOICE AND COLLECTION COMPARISON
# ============================================================

invoice_monthly = (
    invoices.dropna(subset=["invoice_date"])
    .assign(month=lambda df: df["invoice_date"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(
        invoices_issued=("invoice_id", "size"),
        invoiced_amount_rs=("amount_rs", "sum")
    )
    .reset_index()
)

payment_monthly = (
    captured.dropna(subset=["payment_date"])
    .assign(month=lambda df: df["payment_date"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(
        captured_payment_rows=("payment_id", "size"),
        captured_payment_amount_rs=("amount_rs", "sum")
    )
    .reset_index()
)

monthly_comparison = invoice_monthly.merge(
    payment_monthly, on="month", how="outer"
).fillna(0).sort_values("month")

save_csv(monthly_comparison, "11_monthly_invoice_payment_comparison.csv")

print("\nMONTHLY COMPARISON")
print(monthly_comparison.to_string(index=False))

# Note: this compares activity by date, not the same invoice cohort.
# Payments in a month can relate to invoices issued in earlier months.

# ============================================================
# 10. PAYMENT DELAYS
# ============================================================

paid_invoices = invoice_reconciliation[
    invoice_reconciliation["first_payment_date"].notna()
].copy()

paid_invoices["days_to_first_payment"] = (
    paid_invoices["first_payment_date"]
    - paid_invoices["invoice_date"]
).dt.total_seconds() / 86400

# Negative values mean the payment timestamp predates the invoice date.
# These records should be reviewed instead of silently corrected.
payment_delay_summary = pd.DataFrame([{
    "invoices_with_a_matched_captured_payment": len(paid_invoices),
    "average_days_to_first_payment":
        paid_invoices["days_to_first_payment"].mean(),
    "median_days_to_first_payment":
        paid_invoices["days_to_first_payment"].median(),
    "negative_payment_delay_count":
        (paid_invoices["days_to_first_payment"] < 0).sum()
}])

save_csv(paid_invoices, "12_invoice_payment_delays.csv")
save_csv(payment_delay_summary, "13_payment_delay_summary.csv")

print("\nPAYMENT DELAY SUMMARY")
print(payment_delay_summary.to_string(index=False))

# ============================================================
# 11. QUARTER-END OUTSTANDING ANALYSIS
# ============================================================

# Include invoices raised during the selected quarter.
quarter_invoices = invoice_reconciliation[
    invoice_reconciliation["invoice_date"].between(
        QUARTER_START, QUARTER_END
    )
].copy()

quarter_invoice_keys = set(quarter_invoices["invoice_key"])

# For each invoice, sum captured payments received by quarter-end.
quarter_payments = matched_captured[
    matched_captured["invoice_key"].isin(quarter_invoice_keys)
].copy()

payments_by_cutoff = (
    quarter_payments[
        quarter_payments["payment_date"] <= QUARTER_END
    ]
    .groupby("invoice_key")["amount_rs"]
    .sum()
)

payments_after_cutoff = (
    quarter_payments[
        quarter_payments["payment_date"] > QUARTER_END
    ]
    .groupby("invoice_key")["amount_rs"]
    .sum()
)

quarter_invoices["paid_by_quarter_end_rs"] = (
    quarter_invoices["invoice_key"]
    .map(payments_by_cutoff)
    .fillna(0)
)

quarter_invoices["paid_after_quarter_end_rs"] = (
    quarter_invoices["invoice_key"]
    .map(payments_after_cutoff)
    .fillna(0)
)

# Outstanding at cutoff is calculated per invoice.
# Negative balances are not treated as outstanding.
quarter_invoices["outstanding_at_quarter_end_rs"] = (
    quarter_invoices["amount_rs"]
    - quarter_invoices["paid_by_quarter_end_rs"]
).clip(lower=0)

# Portion of that outstanding balance subsequently covered by payments.
quarter_invoices["later_payment_applied_rs"] = quarter_invoices[
    ["outstanding_at_quarter_end_rs", "paid_after_quarter_end_rs"]
].min(axis=1)

quarter_summary = pd.DataFrame([{
    "quarter_start": str(QUARTER_START.date()),
    "quarter_end": str(QUARTER_END.date()),
    "invoices_issued": len(quarter_invoices),
    "amount_invoiced_rs": quarter_invoices["amount_rs"].sum(),
    "payments_received_by_quarter_end_rs":
        quarter_invoices["paid_by_quarter_end_rs"].sum(),
    "outstanding_at_quarter_end_rs":
        quarter_invoices["outstanding_at_quarter_end_rs"].sum(),
    "later_payments_applied_to_outstanding_rs":
        quarter_invoices["later_payment_applied_rs"].sum()
}])

save_csv(quarter_invoices, "14_quarter_invoice_detail.csv")
save_csv(quarter_summary, "15_quarter_end_summary.csv")

print("\nQUARTER-END SUMMARY")
print(quarter_summary.to_string(index=False))

# ============================================================
# 12. BREAKDOWN BY CITY AND CHANNEL
# ============================================================

for dimension in ["city", "channel"]:
    if dimension in invoice_reconciliation.columns:
        breakdown = (
            invoice_reconciliation.groupby(dimension, dropna=False)
            .agg(
                invoice_count=("invoice_id", "size"),
                invoiced_amount_rs=("amount_rs", "sum"),
                captured_amount_rs=("captured_amount_rs", "sum"),
                outstanding_rs=("outstanding_rs", "sum"),
                overpayment_rs=("overpayment_rs", "sum")
            )
            .reset_index()
            .sort_values("outstanding_rs", ascending=False)
        )

        save_csv(
            breakdown,
            f"16_{dimension}_reconciliation.csv"
        )

        print(f"\nRECONCILIATION BY {dimension.upper()}")
        print(breakdown.to_string(index=False))

# ============================================================
# 13. OUTSTANDING AMOUNTS BY INVOICE MONTH
# ============================================================

dated_invoices = invoice_reconciliation.dropna(
    subset=["invoice_date"]
).copy()

dated_invoices["invoice_month"] = (
    dated_invoices["invoice_date"].dt.to_period("M").astype(str)
)

outstanding_by_month = (
    dated_invoices.groupby("invoice_month")
    .agg(
        invoice_count=("invoice_id", "size"),
        invoiced_amount_rs=("amount_rs", "sum"),
        captured_amount_rs=("captured_amount_rs", "sum"),
        outstanding_rs=("outstanding_rs", "sum"),
        overpayment_rs=("overpayment_rs", "sum")
    )
    .reset_index()
    .sort_values("invoice_month")
)

save_csv(outstanding_by_month, "17_outstanding_by_invoice_month.csv")

# ============================================================
# 14. DATA QUALITY CHECKS
# ============================================================

data_quality = pd.DataFrame([
    {
        "check": "Invoices with blank invoice ID",
        "count": invoices["invoice_id"].isna().sum()
    },
    {
        "check": "Duplicate invoice IDs",
        "count": invoices[
            invoices["invoice_id"].notna()
            & invoices["invoice_key"].duplicated(keep=False)
        ]["invoice_id"].nunique()
    },
    {
        "check": "Invoices with invalid invoice date",
        "count": invoices["invoice_date"].isna().sum()
    },
    {
        "check": "Payments with blank payment ID",
        "count": payments["payment_id"].isna().sum()
    },
    {
        "check": "Payments with invalid payment timestamp",
        "count": payments["payment_date"].isna().sum()
    },
    {
        "check": "Captured payments with blank invoice reference",
        "count": (
            captured["invoice_key"] == ""
        ).sum()
    },
    {
        "check": "Settlements with invalid timestamp",
        "count": settlements["settlement_date"].isna().sum()
    },
    {
        "check": "Settlements with blank settlement ID",
        "count": settlements["settlement_key"].eq("").sum()
    }
])

save_csv(data_quality, "18_data_quality_checks.csv")

print("\nDATA QUALITY CHECKS")
print(data_quality.to_string(index=False))

# ============================================================
# 15. FINAL HIGH-LEVEL SUMMARY
# ============================================================

final_summary = pd.DataFrame([
    {
        "metric": "Total invoice rows",
        "value": len(invoices)
    },
    {
        "metric": "Total invoice amount (Rs)",
        "value": invoices["amount_rs"].sum()
    },
    {
        "metric": "All captured payment rows",
        "value": len(captured)
    },
    {
        "metric": "All captured payment amount (Rs)",
        "value": captured["amount_rs"].sum()
    },
    {
        "metric": "Captured payments linked to invoices (Rs)",
        "value": invoice_reconciliation["captured_amount_rs"].sum()
    },
    {
        "metric": "Unmatched captured payment amount (Rs)",
        "value": unmatched_captured["amount_rs"].sum()
    },
    {
        "metric": "Outstanding invoice balances (Rs)",
        "value": invoice_reconciliation["outstanding_rs"].sum()
    },
    {
        "metric": "Overpayment amount (Rs)",
        "value": invoice_reconciliation["overpayment_rs"].sum()
    },
    {
        "metric": "Gateway fees (Rs)",
        "value": settlements["fees_rs"].sum()
    },
    {
        "metric": "Gateway taxes (Rs)",
        "value": settlements["tax_rs"].sum()
    },
    {
        "metric": "Settlement amount recorded (Rs)",
        "value": settlements["amount_rs"].sum()
    }
])

save_csv(final_summary, "19_final_summary.csv")

print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(final_summary.to_string(index=False))

print("\nAnalysis completed.")
print(f"All report files are in: {OUTPUT_FOLDER.resolve()}")
